In [ ]:
%pip install -q requests pandas python-dotenv

In [ ]:
%pip install docling pandas

In [ ]:
from pathlib import Path
from importlib.metadata import version
from IPython.display import display, Markdown, Math
import sys

PDF = Path("/Users/victormoraes/Documents/ia-na-pratica/explorador-de-artigos/artigos/1706.03762v7.pdf").expanduser()

# Começamos com três páginas.
# Depois, escolha um intervalo que contenha fórmulas, tabelas e figuras.
PAGINAS = (1, 7)

if not PDF.is_file():
    raise FileNotFoundError(f"PDF não encontrado: {PDF}")

if PDF.suffix.lower() != ".pdf":
    raise ValueError("Selecione um arquivo PDF.")

if not 1 <= PAGINAS[0] <= PAGINAS[1]:
    raise ValueError("Intervalo de páginas inválido.")

print("Python:", sys.version.split()[0])
print("Docling:", version("docling"))
print("Arquivo:", PDF.name)
print("Tamanho:", round(PDF.stat().st_size / 1024**2, 2), "MB")
print("Páginas selecionadas:", PAGINAS)

In [ ]:
from time import perf_counter

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

USAR_OCR = False
EXTRAIR_FORMULAS = True

opcoes = PdfPipelineOptions()
opcoes.do_ocr = USAR_OCR
opcoes.do_table_structure = True
opcoes.do_formula_enrichment = EXTRAIR_FORMULAS

opcoes.generate_page_images = True
opcoes.generate_picture_images = True
opcoes.images_scale = 2.0

# As figuras serão extraídas com suas legendas.
opcoes.do_picture_description = False

conversor = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=opcoes)
    }
)

inicio = perf_counter()

resultado = conversor.convert(PDF, page_range=PAGINAS)
documento = resultado.document

tempo = perf_counter() - inicio

print("Status:", resultado.status.value)
print(f"Tempo total: {tempo:.1f} segundos")
print("Páginas processadas:", sorted(documento.pages))
print("Tabelas identificadas:", len(documento.tables))
print("Figuras identificadas:", len(documento.pictures))

if resultado.errors:
    print("Problemas reportados:")
    for erro in resultado.errors:
        print(erro)

In [ ]:
print(PAGINAS[0])

In [ ]:
PAGINA_VISUALIZAR = 7

pagina = documento.pages.get(PAGINA_VISUALIZAR)

if pagina is None:
    raise ValueError("Essa página não está entre as páginas processadas.")

print(f"Página original: {PAGINA_VISUALIZAR}")

if pagina.image is not None:
    display(pagina.image.pil_image)
else:
    print("Imagem da página indisponível.")

print("\nConteúdo extraído dessa página:")

markdown_pagina = documento.export_to_markdown(
    page_no=PAGINA_VISUALIZAR
)

display(Markdown(markdown_pagina))

In [ ]:
from docling_core.types.doc import DocItemLabel

formulas = [
    item
    for item, _ in documento.iterate_items()
    if item.label == DocItemLabel.FORMULA
]

print("Blocos identificados como fórmula:", len(formulas))

for numero, formula in enumerate(formulas[:3], start=1):
    paginas = sorted({origem.page_no for origem in formula.prov})

    print(f"\nFórmula {numero} — páginas {paginas}")

    recorte = formula.get_image(documento)
    if recorte is not None:
        display(recorte)

    print("LaTeX reconhecido:")
    print(formula.text)

    if formula.text.strip():
        display(Math(formula.text))

In [ ]:
print("TABELAS")

for numero, tabela in enumerate(documento.tables[:2], start=1):
    paginas = sorted({origem.page_no for origem in tabela.prov})

    print(f"\nTabela {numero} — páginas {paginas}")

    recorte = tabela.get_image(documento)
    if recorte is not None:
        display(recorte)

    df_tabela = tabela.export_to_dataframe(doc=documento)
    display(df_tabela)

if not documento.tables:
    print("Nenhuma tabela identificada no intervalo.")


print("\nFIGURAS")

for numero, figura in enumerate(documento.pictures[:2], start=1):
    paginas = sorted({origem.page_no for origem in figura.prov})

    print(f"\nFigura {numero} — páginas {paginas}")

    imagem = figura.get_image(documento)
    if imagem is not None:
        display(imagem)

    legenda = figura.caption_text(documento)
    print("Legenda:", legenda or "Não identificada.")

if not documento.pictures:
    print("Nenhuma figura identificada no intervalo.")

In [ ]:
import hashlib
import json

from docling_core.types.doc import ImageRefMode

hash_pdf = hashlib.sha256(PDF.read_bytes()).hexdigest()

pasta_saida = (
    Path.cwd()
    / "testes_extracao"
    / f"{hash_pdf[:12]}_p{PAGINAS[0]}-{PAGINAS[1]}"
)
pasta_saida.mkdir(parents=True, exist_ok=True)

documento.save_as_json(
    pasta_saida / "documento.json",
    image_mode=ImageRefMode.REFERENCED,
)

documento.save_as_markdown(
    pasta_saida / "documento.md",
    image_mode=ImageRefMode.REFERENCED,
)

metadados_teste = {
    "arquivo_original": PDF.name,
    "sha256": hash_pdf,
    "paginas_solicitadas": list(PAGINAS),
    "paginas_processadas": sorted(documento.pages),
    "docling_version": version("docling"),
    "ocr": USAR_OCR,
    "reconhecimento_formulas": EXTRAIR_FORMULAS,
    "status": resultado.status.value,
    "tempo_segundos": round(tempo, 2),
}

(pasta_saida / "teste.json").write_text(
    json.dumps(metadados_teste, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Resultados salvos em:")
print(pasta_saida.resolve())